[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_5_advanced_analytics/assignments/python_scripting.ipynb)

# Lab: Python scripting in GRASS

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

This optional lab shows how to drive GRASS from Python so that the steps of the earlier assignments can be repeated on many files and many maps without clicking through them one by one. We start with the Python API itself, then import a folder of lidar tiles with a loop that skips the tiles outside the region of interest, interpolate a DEM on several cores, compute every pairwise difference of a set of DSMs, generate an HTML report with statistics and images, plot terrain profiles with matplotlib, and finally turn one of the steps into a standalone script with its own command line interface.

The data are the Lake Wheeler project and the 2013 Mid Pines lidar you used in Topics 4 and 5. Everything runs in Google Colab (badge above) or locally with GRASS 8.5. The [lab page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_5_advanced_analytics/assignments/lab_python_scripting.html) walks through the same steps.

## Learning objectives

- Call GRASS tools from Python with `grass.tools` and read their output as text, key-value pairs, or JSON
- Loop over files and maps, checking extents before importing
- Use the `nprocs` option to run a tool on several cores
- Collect statistics into a pandas table and write an HTML report
- Sample profiles with `r.profile` and plot them with matplotlib
- Package a workflow as a GRASS script with a parser header

## Outline

- Part 1: Environment setup
- Part 2: The Python API
- Part 3: Import many point clouds
- Part 4: Parallel interpolation
- Part 5: Many maps at once
- Part 6: A generated report
- Part 7: Terrain profiles
- Part 8: From notebook to script
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab). We also install the `pdal` command line tool, used once in Part 3 to tile a point cloud.

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 and the pdal command line tool from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5 pdal

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import pandas, numpy, and matplotlib for the tables and plots.

In [ ]:
import base64
import io
import os
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB); its `PERMANENT` mapset holds the 2015 UAS DSMs
- The 2013 lidar point cloud of Mid Pines, `mid_pines_spm_2013.las` (165 MB)
- The 2013 lidar DEM and DSM of Mid Pines as GRASS raster packs (zip, about 90 MB)

The downloads total about 480 MB and take a minute or two on Colab.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
LAS_URL = f"{BUCKET}/lidar-data/Midpines/mid_pines_spm_2013.las"
LIDAR_URL = f"{BUCKET}/lidar-data/Midpines/secref_mid_pines_spm_elev.zip"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o mid_pines_spm_2013.las {LAS_URL}
!curl -sSL -o secref_mid_pines_spm_elev.zip {LIDAR_URL} && unzip -q -o secref_mid_pines_spm_elev.zip
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `python` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.g_region(...)` runs `g.region`, `tools.r_univar(...)` runs `r.univar`, and so on; a flag like `-p` is passed as `flags="p"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "python"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session, overwrite=True)

print(tools.g_proj(flags="p").text)

### A helper for maps

`show` draws one raster in the current region with a legend and a scale bar, optionally draped over a shaded relief.

In [ ]:
def show(raster, title=None, legend=None, width=500, height=500, flags="sb", shade=None, filename=None):
    """Draw one raster in the current region, optionally draped over a shaded relief."""
    m = gj.Map(width=width, height=height, use_region=True, filename=filename)
    if shade:
        m.d_shade(shade=shade, color=raster)
    else:
        m.d_rast(map=raster)
    if legend:
        m.d_legend(raster=raster, at=(5, 45, 3, 6), title=legend, flags=flags, bgcolor="white")
    m.d_barscale(at=(55, 5))
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()
    return m

---
## Part 2: The Python API

### Calling tools

A GRASS tool is called as a method of the `Tools` object: the dots in the tool name become underscores, parameters become keyword arguments, and flags go into `flags`. Here we set the computational region to the June 2015 DSM and smooth it with a 15 by 15 moving average (`r.neighbors`). Compare with the command line form `r.neighbors -c input=... output=... method=average size=15`.

In [ ]:
tools.g_region(raster="2015_06_20_DSM_agi_11GCP@PERMANENT")
tools.r_neighbors(input="2015_06_20_DSM_agi_11GCP@PERMANENT", output="dsm_smoothed", method="average", size=15, flags="c");

### Reading output

Tools that print something return a result object. `.text` is the raw text, `.keyval` parses `key=value` lines into a dictionary (the `-g` flag of many tools), and `format="json"` returns the parsed JSON with numbers already converted. The JSON form is the one to use in scripts.

In [ ]:
print(tools.g_region(flags="p").text)

In [ ]:
stats = tools.r_univar(map="dsm_smoothed", flags="g").keyval
print(stats)
print("mean:", stats["mean"])

In [ ]:
stats = tools.r_univar(map="dsm_smoothed", format="json").json
print(f"{stats['n']:,} cells, mean {stats['mean']:.2f} m, standard deviation {stats['stddev']:.2f} m")

`g.proj -g` describes the coordinate reference system of the project; `.keyval` turns it into a dictionary.

In [ ]:
proj = tools.g_proj(flags="g").keyval
proj

### Text input

Tools that read from standard input (`input=-` on the command line) take an `io.StringIO` object in Python. Here `v.in.ascii` creates a vector map with two points, one on the road and one in a field, from a string.

In [ ]:
points = """637160.45|219373.24|road
637072.44|219400.61|field
"""
tools.v_in_ascii(input=io.StringIO(points), output="sample_points", separator="pipe",
                 columns="x double precision, y double precision, label text")
print(tools.v_info(map="sample_points", format="json").json["points"], "points")

`r.what` samples rasters at coordinates; with `format="json"` the result is a list with one entry per point.

In [ ]:
for rec in tools.r_what(map="2015_06_20_DSM_agi_11GCP@PERMANENT", coordinates=(637160.45, 219373.24, 637072.44, 219400.61), format="json").json:
    print(rec["easting"], rec["northing"], rec["2015_06_20_DSM_agi_11GCP@PERMANENT"]["value"])

### Map algebra, the region, and listing maps

`r.mapcalc` takes the expression as a string. The region settings come back as JSON with `g.region`, which is how a script learns the resolution or the cell area. `g.list` with `format="json"` returns one record per map with its name and mapset.

In [ ]:
tools.r_mapcalc(expression="dsm_above_115 = if(2015_06_20_DSM_agi_11GCP@PERMANENT > 115, 2015_06_20_DSM_agi_11GCP@PERMANENT, null())")
print(tools.r_univar(map="dsm_above_115", format="json").json["n"], "cells above 115 m")

In [ ]:
region = tools.g_region(flags="p", format="json").json
print(f"{region['rows']} rows, {region['cols']} columns, cell area {region['nsres'] * region['ewres']} m2")

In [ ]:
for item in tools.g_list(type="raster", pattern="2015_06_*", mapset="PERMANENT", format="json").json:
    print(item["fullname"])

current_mapset = tools.g_gisenv(get="MAPSET").text.strip()
print("current mapset:", current_mapset)

### The classic `grass.script` functions

Older scripts and most manual pages use `grass.script`, imported as `gs`. `run_command` runs a tool, `read_command` returns its text, `parse_command` parses `key=value` output, and `write_command` passes text on standard input. They take the same parameters; `grass.tools` only wraps them with nicer return values. You will meet both forms.

In [ ]:
gs.run_command("g.region", raster="2015_06_20_DSM_agi_11GCP@PERMANENT")
print(gs.read_command("g.region", flags="g"))
print(gs.parse_command("r.univar", map="dsm_smoothed", flags="g")["mean"])
gs.write_command("v.in.ascii", input="-", stdin="637080|219480|road2", output="one_point", separator="pipe", overwrite=True)

### PyGRASS: working with vector features

`grass.tools` runs whole tools. To create or read individual features, PyGRASS gives Python classes backed by the GRASS C library. Here we write two points with an attribute table, then read them back.

In [ ]:
from grass.pygrass.vector import VectorTopo
from grass.pygrass.vector.geometry import Point

cols = [("cat", "INTEGER PRIMARY KEY"), ("name", "TEXT")]
with VectorTopo("places", mode="w", tab_cols=cols, overwrite=True) as places:
    places.write(Point(637160.45, 219373.24), cat=1, attrs=("road",))
    places.write(Point(637072.44, 219400.61), cat=2, attrs=("field",))
    places.table.conn.commit()

with VectorTopo("places", mode="r") as places:
    for point in places:
        print(point, point.attrs["name"])

---
## Part 3: Import many point clouds

Lidar deliveries come as tiles, dozens or hundreds of files that cover a county. To build a surface for a small study area, a script has to find the tiles that touch the area, import only those, and merge them. We make such a tile set from the 2013 Mid Pines point cloud with the `pdal tile` command: 500 m tiles aligned to the lower left corner of the data. `r.in.pdal -g` prints the extent of a file without importing it; `gs.parse_key_val` turns the one-line `n=... s=...` output into a dictionary.

In [ ]:
ext = gs.parse_key_val(tools.r_in_pdal(input="mid_pines_spm_2013.las", output="dummy", flags="g").text, vsep=" ", val_type=float)
print(ext)
Path("tiles").mkdir(exist_ok=True)

In [ ]:
!pdal tile --length 500 --origin_x {ext["w"]} --origin_y {ext["s"]} mid_pines_spm_2013.las "tiles/tile_#.las"
!ls -lh tiles

### The import loop

The study area is the field block of Assignment 5A. For every tile we read its extent and compare it with the region: a tile is skipped when it lies entirely north, south, east, or west of the region. The tiles that touch the region are imported with `v.in.pdal`, limited to the region (`-r`) and to the ground class (`class_filter=2`), and `v.build` adds the topology that `v.patch` needs. The imported tiles are merged with `v.patch` and removed.

In [ ]:
tools.g_region(n=219637, s=219254, w=636730, e=637193, res=1, flags="p")
region = tools.g_region(flags="g").keyval

imported = []
for las in sorted(Path("tiles").glob("*.las")):
    ext = gs.parse_key_val(tools.r_in_pdal(input=str(las), output="dummy", flags="g").text, vsep=" ", val_type=float)
    if ext["n"] < region["s"] or ext["s"] > region["n"] or ext["e"] < region["w"] or ext["w"] > region["e"]:
        print(f"skipping {las.name}")
        continue
    name = las.stem
    tools.v_in_pdal(input=str(las), output=name, flags="r", class_filter=2)
    tools.v_build(map=name)
    n = tools.v_info(map=name, format="json").json["points"]
    print(f"{las.name}: {n:,} ground points in the region")
    imported.append(name)

tools.v_patch(input=imported, output="ground_2013")
tools.g_remove(type="vector", name=imported, flags="f")
print(tools.v_info(map="ground_2013", format="json").json["points"], "ground points merged")

#### Map: the merged ground points

A density raster from `r.in.pdal` is the quickest look at a point cloud; here the number of ground points per 1 m cell, read directly from the two tiles listed in a text file. The tiles written by `pdal tile` store their coordinate system in a form that `r.in.pdal` does not recognize, so the `-o` flag tells it to trust the project's CRS.

In [ ]:
Path("tiles.txt").write_text("\n".join(f"tiles/{name}.las" for name in imported) + "\n")
tools.r_in_pdal(file="tiles.txt", output="ground_density", method="n", class_filter=2, flags="o")
tools.r_colors(map="ground_density", color="viridis")
show("ground_density", title="Ground points per cell", legend="points")

---
## Part 4: Parallel interpolation

`v.surf.rst` interpolates the ground points into a 1 m DEM. Since GRASS 8.3 the tool has an `nprocs` option that splits the work over several cores; the result is the same, only faster. We time the run with one core and with all the cores of the machine (Colab usually gives two).

In [ ]:
cores = os.cpu_count()
for n in (1, cores):
    start = time.time()
    tools.v_surf_rst(input="ground_2013", elevation="dem_2013_1m", tension=30, smooth=1, npmin=100, nprocs=n)
    print(f"nprocs={n}: {time.time() - start:.1f} s")

In [ ]:
tools.r_relief(input="dem_2013_1m", output="dem_2013_1m_relief")
tools.r_colors(map="dem_2013_1m", color="elevation")
show("dem_2013_1m", shade="dem_2013_1m_relief", title="Lidar ground DEM, 1 m", legend="m")

Tools without an `nprocs` option can still be run in parallel by tiling the region and running one process per tile; PyGRASS provides `GridModule` for that:

```python
from grass.pygrass.modules.grid import GridModule

grd = GridModule("v.surf.rst", width=250, height=250, overlap=20, processes=4,
                 input="ground_2013", elevation="dem_2013_1m", tension=30, smooth=1, npmin=100)
grd.run()
```

Tiling costs time for patching the tiles back together, and it only works for tools whose result in one cell depends on a neighborhood of bounded size (the `overlap`). A simulation such as `r.sim.water`, where water flows across the whole area, cannot be tiled this way; it has its own `nprocs` option instead.

---
## Part 5: Many maps at once

In Topic 4 you compared the DSMs of the June 2015 flight produced by three software packages. Two nested loops produce every pairwise difference. The name of the difference raster is built from the names of the inputs (an `@mapset` suffix is not allowed in a new name, so it is replaced), `r.mapcalc` computes the difference, and `r.colors` gives each result the `differences` color table.

In [ ]:
rasters = ["2015_06_20_pix4d_11GCP_dsm", "2015_06_20_DSM_Trimble_11GCP", "2015_06_20_DSM_agi_11GCP"]
tools.g_region(raster=rasters[0])

for raster_a in rasters:
    for raster_b in rasters:
        if raster_a == raster_b:
            continue
        difference = f"diff_{raster_a.replace('@', '_')}_{raster_b.replace('@', '_')}"
        tools.r_mapcalc(expression=f"{difference} = {raster_a} - {raster_b}")
        tools.r_colors(map=difference, color="differences")
        stats = tools.r_univar(map=difference, format="json").json
        print(f"{difference}: mean {stats['mean']:6.2f} m, stddev {stats['stddev']:5.2f} m")

In [ ]:
show("diff_2015_06_20_pix4d_11GCP_dsm_2015_06_20_DSM_agi_11GCP", title="Pix4D minus Agisoft", legend="m")

Loops create many maps; `g.remove` with a pattern removes them again.

In [ ]:
tools.g_remove(type="raster", pattern="diff_*", flags="f");

---
## Part 6: A generated report

A report for a set of maps needs the same things for each map: statistics, a shaded relief, and an image. We list the June 2015 DSMs with `g.list`, loop over them, collect the statistics in a dictionary, compute the relief, and render each map to a PNG with `gj.Map`, which writes the file named in `filename`.

In [ ]:
rasters = [item["fullname"] for item in tools.g_list(type="raster", pattern="2015_06_*", mapset="PERMANENT", format="json").json]
tools.g_region(raster=rasters[0])

stats = {}
for raster in rasters:
    stats[raster] = tools.r_univar(map=raster, format="json").json
    relief = raster.replace("@", "_") + "_relief"
    tools.r_relief(input=raster, output=relief)
    tools.r_colors(map=raster, color="elevation")
    m = gj.Map(width=400, height=400, use_region=True, filename=raster.replace("@", "_") + ".png")
    m.d_shade(shade=relief, color=raster)
    m.d_legend(raster=raster, at=(5, 45, 3, 6), flags="sb", bgcolor="white")
print("\n".join(rasters))

The dictionary of statistics becomes a pandas table, which is easy to inspect, sort, and save as CSV.

In [ ]:
table = pd.DataFrame(stats).T[["n", "min", "max", "mean", "stddev"]]
table.to_csv("dsm_stats.csv")
table.round(2)

The HTML report is a template filled in once per map. The images are embedded in the file so that it can be opened anywhere; `IPython.display.HTML` shows it in the notebook, and the file `report.html` can be downloaded and printed to PDF.

In [ ]:
template = """<h2>Raster map {name}</h2>
<table border="1" cellpadding="4">
<tr><td>Cells</td><td>{n:,}</td></tr>
<tr><td>Minimum</td><td>{min:.2f} m</td></tr>
<tr><td>Maximum</td><td>{max:.2f} m</td></tr>
<tr><td>Mean</td><td>{mean:.2f} m</td></tr>
<tr><td>Standard deviation</td><td>{stddev:.2f} m</td></tr>
</table>
<img src="data:image/png;base64,{image}">
"""

with open("report.html", "w") as output:
    output.write("<h1>June 2015 DSMs of Mid Pines</h1>")
    for raster in sorted(rasters):
        image = base64.b64encode(Path(raster.replace("@", "_") + ".png").read_bytes()).decode()
        output.write(template.format(name=raster, image=image, **stats[raster]))

HTML(Path("report.html").read_text())

---
## Part 7: Terrain profiles

`r.profile` samples a raster along a line given by its vertices. We sample the October 2015 UAS DSM and the 2013 lidar DEM along the same line, which crosses the road and a field, read the output into pandas, and plot both. The lidar DEM comes from the raster pack you used in Assignment 5A.

In [ ]:
tools.r_unpack(input="mid_pines_lidar2013_dem.pack", flags="o")
tools.g_region(raster="2015_10_06_DSM_agi_8GCPs@PERMANENT")

coordinates = [637160.446919, 219373.236976, 637105.693795, 219392.416036, 637072.439779, 219400.613538]
profiles = {}
for raster in ["2015_10_06_DSM_agi_8GCPs@PERMANENT", "mid_pines_lidar2013_dem"]:
    text = tools.r_profile(input=raster, coordinates=coordinates, flags="g").text
    profiles[raster] = pd.read_csv(io.StringIO(text), sep=r"\s+", names=["x", "y", "distance", "elevation"])
profiles["mid_pines_lidar2013_dem"].head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for raster, profile in profiles.items():
    ax.plot(profile["distance"], profile["elevation"], label=raster.split("@")[0])
ax.set_xlabel("Distance along the line (m)")
ax.set_ylabel("Elevation (m)")
ax.legend()
plt.tight_layout()
plt.show()

With the elevations in a numpy array we can describe each profile: range, mean, median, spread, and a simple roughness measure, the standard deviation of the differences between consecutive samples.

In [ ]:
rows = []
for raster, profile in profiles.items():
    z = profile["elevation"].to_numpy()
    rows.append({"raster": raster.split("@")[0], "min": z.min(), "max": z.max(), "mean": z.mean(),
                 "median": np.median(z), "stddev": z.std(), "roughness": np.std(np.diff(z))})
profile_stats = pd.DataFrame(rows).set_index("raster")
profile_stats.to_csv("profile_stats.csv")
profile_stats.round(3)

---
## Part 8: From notebook to script

A notebook is for exploring. Once the steps are settled, the same code becomes a script that can be run again on new data, from the command line, with parameters instead of edited constants. A GRASS script declares its parameters in a comment header; `gs.parser()` reads them from the command line and the same header gives the script a graphical dialog in the GUI. The cell below writes the script to a file.

In [ ]:
%%writefile dsm_stats.py
#!/usr/bin/env python3

# %module
# % description: Writes univariate statistics of the rasters matching a pattern to a CSV file.
# % keyword: raster
# % keyword: statistics
# %end
# %option
# % key: pattern
# % type: string
# % description: Pattern of raster names (wildcards allowed)
# % required: yes
# %end
# %option G_OPT_F_OUTPUT
# % key: output
# % description: Name of the CSV file to write
# %end
# %option G_OPT_M_MAPSET
# % key: mapset
# % description: Restrict the search to this mapset (default: all mapsets)
# % required: no
# %end

import csv
import sys

import grass.script as gs
from grass.tools import Tools


def main():
    options, flags = gs.parser()
    tools = Tools()
    search = {"mapset": options["mapset"]} if options["mapset"] else {}
    rasters = [item["fullname"] for item in tools.g_list(type="raster", pattern=options["pattern"], format="json", **search).json]
    with open(options["output"], "w", newline="") as output:
        writer = csv.writer(output)
        writer.writerow(["raster", "n", "min", "max", "mean", "stddev"])
        for raster in rasters:
            s = tools.r_univar(map=raster, format="json").json
            writer.writerow([raster, s["n"], s["min"], s["max"], s["mean"], s["stddev"]])
    gs.message(f"Wrote statistics of {len(rasters)} rasters to {options['output']}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

`grass <mapset> --exec <command>` runs a command inside a GRASS session without opening a shell or the GUI, so the script runs from any terminal. Asking for `--help` shows the interface the header generated.

In [ ]:
!grass {mapset} --exec python3 dsm_stats.py --help

In [ ]:
!grass {mapset} --exec python3 dsm_stats.py pattern="2015_06_*" mapset=PERMANENT output=dsm_stats_script.csv
pd.read_csv("dsm_stats_script.csv").round(2)

In the GRASS GUI the same code runs from the *Python* tab of the console (one statement at a time) or from the *Simple Python Editor* (*File > Simple Python Editor*), which runs a whole file. A script with a parser header can also be installed with `g.extension` and then appears as a tool of its own.

---
## Report questions

Answer in the cells below, or in your report document. Keep the answers short and refer to the outputs above.

#### Question 1: Import loop

Which tiles were imported and which were skipped, and how many ground points ended up in the merged map? What would change in the loop if the tiles were delivered as LAZ files or if you wanted all returns rather than the ground class?

`Add text here with your answer to question 1`

#### Question 2: Parallel interpolation

Report the two run times and the number of cores. Why is the speedup less than the number of cores, and which step in this lab could not be sped up by tiling?

`Add text here with your answer to question 2`

#### Question 3: Generated report

Extend the report loop with one more product per raster, for example a viewshed from a point of your choice (`r.viewshed`) or a slope map, and include the result. Which lines did you change?

`Add text here with your answer to question 3`

#### Question 4: Profiles

Run the profile cell with a line of your own and describe what the two profiles show: where do the UAS DSM and the lidar DEM agree, where do they differ, and what does the roughness value capture?

`Add text here with your answer to question 4`